# NHANES XPT Fidelity Check & CSV Converter

This notebook converts **one original NHANES `.xpt` file** into a corrected CSV while keeping an audit trail.

It performs these checks **before and after correction**:

- Detects the known tiny-value representation `5.397605346934028e-79` in numeric variables.
- Replaces **only that exact value** with a true numeric `0`.
- If `HUQ050` is present, checks the final zero count against the expected CDC count of **1,250**.
- If `SMDUPCA` is present, removes numeric `.0` artifacts and restores the field to a **12-character UPC** using left-zero padding.
- Reports every variable affected by the zero correction.
- Writes a corrected CSV and a separate audit-report CSV.
- Re-reads the exported CSV to verify that the corrected values survived export.

> The original `.xpt` file is never modified.


## 1. Set the input XPT file

Change only `INPUT_XPT` below, then run the notebook from top to bottom.

Example on Windows:

```python
INPUT_XPT = Path(r"C:\NHANES\2003-2004\HUQ_C.xpt")
```


In [ ]:
from pathlib import Path
import csv
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# EDIT THIS PATH
# ============================================================
INPUT_XPT = Path(r"C:\path\to\your\file.xpt")

# Output will be written beside the XPT in this subfolder.
OUTPUT_DIR = INPUT_XPT.parent / "converted_csv"

# Exact value observed when some SAS XPT numeric zeros are decoded incorrectly.
ZERO_SENTINEL = np.float64(5.397605346934028e-79)

# Optional source/codebook expectations.
# Add more variables here if you have authoritative expected zero counts.
EXPECTED_ZERO_COUNTS = {
    "HUQ050": 1250,
}

# Character fields that must retain a fixed width.
# Add other known fixed-width character variables if needed.
CHARACTER_WIDTHS = {
    "SMDUPCA": 12,
}

print("Input:", INPUT_XPT)
print("Output folder:", OUTPUT_DIR)


## 2. Helper functions

In [ ]:
def exact_sentinel_counts(df, sentinel=ZERO_SENTINEL):
    """Return counts of the exact bad tiny value in every numeric column."""
    rows = []
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    for col in numeric_cols:
        count = int(df[col].eq(sentinel).sum())
        if count:
            rows.append({
                "variable": col,
                "sentinel_count": count,
                "true_zero_count_before": int(df[col].eq(0).sum()),
            })

    if not rows:
        return pd.DataFrame(
            columns=["variable", "sentinel_count", "true_zero_count_before"]
        )

    return (
        pd.DataFrame(rows)
        .sort_values(["sentinel_count", "variable"], ascending=[False, True])
        .reset_index(drop=True)
    )


def normalize_fixed_width_character(series, width):
    """
    Preserve missing values, remove a trailing .0 artifact, and restore
    leading zeros only for digit-only values.
    """
    s = series.astype("string").str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})

    # Remove one or more zeroes after a decimal point: 123.0 -> 123
    s = s.str.replace(r"\.0+$", "", regex=True)

    digit_mask = s.notna() & s.str.fullmatch(r"\d+")
    s.loc[digit_mask] = s.loc[digit_mask].str.zfill(width)

    return s


def character_audit(series, width):
    """Audit one expected fixed-width character field."""
    s = series.astype("string")

    nonmissing = s.dropna()
    ends_dot_zero = int(nonmissing.str.contains(r"\.0+$", regex=True).sum())
    wrong_length = int(nonmissing.str.len().ne(width).sum())
    nondigit = int((~nonmissing.str.fullmatch(r"\d+")).sum())

    return {
        "nonmissing": int(nonmissing.shape[0]),
        "ends_in_dot_zero": ends_dot_zero,
        "wrong_length": wrong_length,
        "non_digit": nondigit,
    }


def value_check(df, variable, max_rows=30):
    """Display value frequencies for a variable, including missing values."""
    if variable not in df.columns:
        print(f"{variable}: not present in this XPT file")
        return

    counts = (
        df[variable]
        .value_counts(dropna=False)
        .rename_axis(variable)
        .reset_index(name="count")
    )
    display(counts.head(max_rows))


def build_audit_report(
    raw_df,
    corrected_df,
    sentinel_audit_before,
    expected_zero_counts=EXPECTED_ZERO_COUNTS,
    character_widths=CHARACTER_WIDTHS,
):
    rows = []

    # Numeric zero/sentinel audit
    for _, rec in sentinel_audit_before.iterrows():
        col = rec["variable"]
        rows.append({
            "issue": "numeric_zero_fidelity",
            "variable": col,
            "before_problem_count": int(rec["sentinel_count"]),
            "after_problem_count": int(corrected_df[col].eq(ZERO_SENTINEL).sum()),
            "final_true_zero_count": int(corrected_df[col].eq(0).sum()),
            "status": (
                "PASS"
                if int(corrected_df[col].eq(ZERO_SENTINEL).sum()) == 0
                else "FAIL"
            ),
            "notes": "Exact sentinel replaced with numeric zero.",
        })

    # Expected zero counts
    for col, expected in expected_zero_counts.items():
        if col in corrected_df.columns:
            final_zeros = int(corrected_df[col].eq(0).sum())
            rows.append({
                "issue": "expected_zero_count",
                "variable": col,
                "before_problem_count": "",
                "after_problem_count": "",
                "final_true_zero_count": final_zeros,
                "status": "PASS" if final_zeros == expected else "FAIL",
                "notes": f"Expected authoritative zero count = {expected}.",
            })

    # Fixed-width character audit
    for col, width in character_widths.items():
        if col in corrected_df.columns:
            before = character_audit(raw_df[col], width)
            after = character_audit(corrected_df[col], width)
            passed = (
                after["ends_in_dot_zero"] == 0
                and after["wrong_length"] == 0
                and after["non_digit"] == 0
            )
            rows.append({
                "issue": "character_fidelity",
                "variable": col,
                "before_problem_count": (
                    before["ends_in_dot_zero"] + before["wrong_length"]
                ),
                "after_problem_count": (
                    after["ends_in_dot_zero"] + after["wrong_length"]
                ),
                "final_true_zero_count": "",
                "status": "PASS" if passed else "FAIL",
                "notes": (
                    f"Expected {width}-digit character value. "
                    f"After: .0={after['ends_in_dot_zero']}, "
                    f"wrong_length={after['wrong_length']}, "
                    f"non_digit={after['non_digit']}."
                ),
            })

    return pd.DataFrame(rows)


## 3. Read the original XPT — no corrections yet

In [ ]:
if not INPUT_XPT.exists():
    raise FileNotFoundError(
        f"XPT file not found: {INPUT_XPT}\n"
        "Update INPUT_XPT in Section 1."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read source exactly once into the raw audit dataframe.
raw_df = pd.read_sas(
    INPUT_XPT,
    format="xport",
    encoding="latin1"
)

print(f"Rows: {len(raw_df):,}")
print(f"Columns: {raw_df.shape[1]:,}")
print("\nFirst 5 rows:")
display(raw_df.head())


## 4. Audit the original XPT read BEFORE changing anything

In [ ]:
sentinel_before = exact_sentinel_counts(raw_df)

total_bad = (
    int(sentinel_before["sentinel_count"].sum())
    if not sentinel_before.empty
    else 0
)

print(f"Exact sentinel occurrences: {total_bad:,}")
print(f"Numeric variables affected: {len(sentinel_before):,}")

if sentinel_before.empty:
    print("No exact zero-sentinel values detected.")
else:
    display(sentinel_before)


In [ ]:
# Detailed checks for the two variables discussed in the NHANES fidelity review.

if "HUQ050" in raw_df.columns:
    print("HUQ050 BEFORE correction")
    print("True numeric zeros:", int(raw_df["HUQ050"].eq(0).sum()))
    print("Tiny sentinel values:", int(raw_df["HUQ050"].eq(ZERO_SENTINEL).sum()))
    value_check(raw_df, "HUQ050")

if "SMDUPCA" in raw_df.columns:
    print("\nSMDUPCA BEFORE correction")
    print(character_audit(raw_df["SMDUPCA"], CHARACTER_WIDTHS["SMDUPCA"]))
    display(raw_df[["SMDUPCA"]].dropna().head(20))


## 5. Create the corrected dataframe

The correction is intentionally narrow:

1. **Only the exact known sentinel** `5.397605346934028e-79` is converted to `0`.
2. Fixed-width character corrections are applied only to variables listed in `CHARACTER_WIDTHS`.
3. No missing-value codes, survey codes, or other numeric values are changed.


In [ ]:
corrected_df = raw_df.copy()

# ------------------------------------------------------------
# A. Correct the exact zero sentinel in numeric columns
# ------------------------------------------------------------
numeric_cols = corrected_df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    mask = corrected_df[col].eq(ZERO_SENTINEL)
    if mask.any():
        corrected_df.loc[mask, col] = 0.0

# ------------------------------------------------------------
# B. Restore declared fixed-width character variables
# ------------------------------------------------------------
for col, width in CHARACTER_WIDTHS.items():
    if col in corrected_df.columns:
        corrected_df[col] = normalize_fixed_width_character(
            corrected_df[col],
            width
        )

print("Corrections applied to an in-memory copy.")


## 6. Validate the corrected dataframe BEFORE export

In [ ]:
sentinel_after = exact_sentinel_counts(corrected_df)
remaining_bad = (
    int(sentinel_after["sentinel_count"].sum())
    if not sentinel_after.empty
    else 0
)

print(f"Remaining exact sentinel occurrences: {remaining_bad:,}")

if remaining_bad != 0:
    raise AssertionError(
        f"Zero fidelity FAILED: {remaining_bad:,} sentinel values remain."
    )

for variable, expected in EXPECTED_ZERO_COUNTS.items():
    if variable in corrected_df.columns:
        actual = int(corrected_df[variable].eq(0).sum())
        status = "PASS" if actual == expected else "FAIL"
        print(
            f"{variable}: true zeros = {actual:,}; "
            f"expected = {expected:,} -> {status}"
        )

for variable, width in CHARACTER_WIDTHS.items():
    if variable in corrected_df.columns:
        audit = character_audit(corrected_df[variable], width)
        print(f"{variable}: {audit}")

print("\nHUQ050 AFTER correction")
value_check(corrected_df, "HUQ050")


## 7. Build the audit report

In [ ]:
audit_df = build_audit_report(
    raw_df=raw_df,
    corrected_df=corrected_df,
    sentinel_audit_before=sentinel_before,
)

if audit_df.empty:
    print("No configured fidelity issues were found in this file.")
else:
    display(audit_df)

failed = audit_df.loc[audit_df["status"].eq("FAIL")] if not audit_df.empty else audit_df

if not failed.empty:
    print("\nWARNING: One or more configured checks FAILED.")
    display(failed)
else:
    print("\nAll configured checks PASS.")


## 8. Export the corrected CSV and audit report

In [ ]:
csv_path = OUTPUT_DIR / f"{INPUT_XPT.stem}_corrected.csv"
audit_path = OUTPUT_DIR / f"{INPUT_XPT.stem}_audit.csv"

corrected_df.to_csv(
    csv_path,
    index=False,
    na_rep="",
    quoting=csv.QUOTE_MINIMAL
)

audit_df.to_csv(
    audit_path,
    index=False
)

print("Corrected CSV:", csv_path)
print("Audit report:", audit_path)


## 9. Re-read the CSV and perform a final round-trip check

For fields such as `SMDUPCA`, the re-read explicitly uses string dtype so that validation checks the actual characters stored in the CSV rather than allowing pandas to infer a numeric dtype.

> Spreadsheet programs such as Excel may visually reinterpret a UPC as a number when opening a CSV. That does not mean the CSV text itself lost the leading zeros. For validation, read character identifiers explicitly as strings.


In [ ]:
dtype_map = {
    col: "string"
    for col in CHARACTER_WIDTHS
    if col in corrected_df.columns
}

roundtrip_df = pd.read_csv(
    csv_path,
    dtype=dtype_map,
    low_memory=False
)

# Check sentinel after CSV round trip.
numeric_roundtrip = roundtrip_df.select_dtypes(include=[np.number])
roundtrip_sentinel_count = int(
    numeric_roundtrip.eq(ZERO_SENTINEL).sum().sum()
)

print("Round-trip sentinel count:", roundtrip_sentinel_count)

if roundtrip_sentinel_count != 0:
    raise AssertionError(
        "CSV round-trip FAILED: tiny zero-sentinel values remain."
    )

# Expected zero checks.
for variable, expected in EXPECTED_ZERO_COUNTS.items():
    if variable in roundtrip_df.columns:
        series = pd.to_numeric(roundtrip_df[variable], errors="coerce")
        actual = int(series.eq(0).sum())
        print(
            f"Round-trip {variable} zeros: {actual:,} "
            f"(expected {expected:,})"
        )
        if actual != expected:
            raise AssertionError(
                f"{variable} expected {expected:,} zeros, "
                f"but CSV contains {actual:,}."
            )

# Fixed-width character checks.
for variable, width in CHARACTER_WIDTHS.items():
    if variable in roundtrip_df.columns:
        audit = character_audit(roundtrip_df[variable], width)
        print(f"Round-trip {variable}: {audit}")
        if (
            audit["ends_in_dot_zero"] != 0
            or audit["wrong_length"] != 0
            or audit["non_digit"] != 0
        ):
            raise AssertionError(
                f"{variable} character fidelity FAILED after CSV export."
            )

print("\nFINAL RESULT: all configured round-trip checks PASS.")
print("Use this CSV:", csv_path)


## How to use this notebook for another NHANES file

1. Change `INPUT_XPT`.
2. Add any authoritative zero-count checks to `EXPECTED_ZERO_COUNTS`.
3. Add any known fixed-width character fields to `CHARACTER_WIDTHS`.
4. Run **Restart Kernel and Run All**.
5. Review the audit table.
6. Use the generated `_corrected.csv` only when the final checks pass.

For a file that does not contain `HUQ050` or `SMDUPCA`, those checks are skipped automatically.
